# Подготовка данных для обучения моделей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators
* https://pytorch.org/docs/stable/data.html 
* https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann


## Задачи для совместного разбора

1. Создайте синтетический датасет для задачи регрессии и представьте его в виде `torch.utils.data.Dataset`

In [1]:
# Импортируем необходимые библиотеки
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Callable, Any
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Создаем синтетический датасет для регрессии
class SyntheticRegressionDataset(Dataset):
    """Синтетический датасет для задачи регрессии"""
    
    def __init__(self, n_samples=1000, n_features=10, noise=0.1):
        # Генерируем синтетические данные
        X, y = make_regression(n_samples=n_samples, 
                              n_features=n_features, 
                              noise=noise, 
                              random_state=42)
        
        # Конвертируем в тензоры PyTorch
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        """Возвращает размер датасета"""
        return len(self.X)
    
    def __getitem__(self, idx):
        """Возвращает один элемент датасета"""
        return self.X[idx], self.y[idx]

# Демонстрация работы
regression_dataset = SyntheticRegressionDataset(n_samples=100, n_features=5)
print(f"Размер датасета: {len(regression_dataset)}")
print(f"Пример элемента: {regression_dataset[0]}")
print(f"Форма признаков: {regression_dataset[0][0].shape}")

Размер датасета: 100
Пример элемента: (tensor([ 0.9751, -0.6772, -0.0122, -0.8973,  0.0758]), tensor(-57.1958))
Форма признаков: torch.Size([5])


## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Считайте файл `bank-full.csv` ([источник](https://www.kaggle.com/datasets/hariharanpavan/bank-marketing-dataset-analysis-classification)) в виде `pd.DataFrame`.

Опишите класс `BankDatasetBase`. Решение должно удовлетворять следующим критериям:

* класс наследуется от `torch.utils.data.Dataset`;
* при создании объекта в конструктор передается набор данных в виде `pd.DataFrame`;
* объекты класса имеют поля `X` и `y` с признаками и метками соответственно;
* класс реализует интерфейс последовательностей (`__getitem__` + `__len__`);
* `obj[i]` возвращает кортеж, содержащий `i`-ую строку из `obj.X` (серию) и `i`-ую строку из `obj.y` (строку).
    
Создайте объект класса `BankDatasetBase` и продемонстрируйте работоспособность.

- [ ] Проверено на семинаре

In [2]:

bank_df = pd.read_csv('bank-full.csv', sep=',')

print(f"Размер датасета: {bank_df.shape}")
print(f"\nНазвания столбцов:")
print(bank_df.columns.tolist())
print("\nПервые 5 строк:")
print(bank_df.head())
    

class BankDatasetBase(Dataset):
    """Базовый класс для работы с банковским датасетом"""
    
    def __init__(self, data: pd.DataFrame) -> None:

        self.X = data.drop('y', axis=1)
        self.y = data['y']              
        
        print(f"Инициализирован датасет с {len(self.X)} образцами и {len(self.X.columns)} признаками")

    def __getitem__(self, idx: int) -> tuple:

        x = self.X.iloc[idx] 
        y = self.y.iloc[idx]  
        return x, y
    
    def __len__(self) -> int:
        """Возвращает количество элементов в датасете"""
        return len(self.X)

bank_dataset_base = BankDatasetBase(bank_df)
print(f"\nРазмер датасета: {len(bank_dataset_base)}")
print(f"\nПример элемента [0]:")
x, y = bank_dataset_base[0]
print(f"Признаки (тип: {type(x)}): {x}")
print(f"Метка: {y}")

Размер датасета: (45211, 17)

Названия столбцов:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Первые 5 строк:
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may   

<p class="task" id="2"></p>

2\. Опишите класс `BankDataset`. Решение должно удовлетворять всем критериям из предыдущего задания, а также:
* при создании объекта в конструктор может быть передан необязательные аргументы `transform` и `target_transform`;
* если аргумент `transform` был передан, то при получении `i`-го элемента, нужно вызвать `transform(x)` и вернуть полученный результат.
* если аргумент `target_transform` был передан, то при получении `i`-го элемента, нужно вызвать `target_transform(y)` и вернуть полученный результат.

Создайте объект класса `BankDataset` и продемонстрируйте работоспособность (без передачи `target_transform` и `transform`).

- [ ] Проверено на семинаре

In [3]:
class BankDataset(Dataset):
    """Расширенный класс для работы с банковским датасетом с поддержкой трансформаций"""
    
    def __init__(
            self, 
            data: pd.DataFrame, 
            transform: Callable | None = None,
            target_transform: Callable | None = None
    ) -> None:


        self.X = data.drop('y', axis=1) if 'y' in data.columns else data
        self.y = data['y'] if 'y' in data.columns else None
        
        self.transform = transform
        self.target_transform = target_transform
        
        print(f"Инициализирован BankDataset с {len(self.X)} образцами")
        if transform is not None:
            print(f"- Используется transform: {type(transform).__name__}")
        if target_transform is not None:
            print(f"- Используется target_transform: {type(target_transform).__name__}")

    def __getitem__(self, idx: int) -> tuple:

        x = self.X.iloc[idx]
        y = self.y.iloc[idx] if self.y is not None else None  
        
        if self.transform is not None:
            x = self.transform(x)
        
        if self.target_transform is not None and y is not None:
            y = self.target_transform(y)
        
        return x, y
    
    def __len__(self) -> int:
        """Возвращает количество элементов в датасете"""
        return len(self.X)


bank_dataset = BankDataset(bank_df)
print(f"\nРазмер датасета: {len(bank_dataset)}")
print(f"\nПример элемента [0] без трансформаций:")
x, y = bank_dataset[0]
print(f"Признаки: {x.head(3)}...") 
print(f"Метка: {y}")

Инициализирован BankDataset с 45211 образцами

Размер датасета: 45211

Пример элемента [0] без трансформаций:
Признаки: age                58
job        management
marital       married
Name: 0, dtype: object...
Метка: no


<p class="task" id="3"></p>

3\. Опишите класс `OrdinalEncoderTransform`. Решение должно удовлетворять следующим критериям:

* при создании объекта в конструктор передаются названия нечисловых столбцов в датасете
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (признаки) и возвращает набор признаков, в котором нечисловые характеристики закодированы целыми числами;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объект класса `OrdinalEncoderTransform`.

- [ ] Проверено на семинаре

In [4]:
class Transform:
    """Базовый класс для всех трансформаций"""
    pass

In [ ]:
class OrdinalEncoderTransform(Transform):
    """Кодировщик категориальных признаков в порядковые числа"""
    
    def __init__(self, category_columns: list[str]) -> None:
        self.category_columns = category_columns
        self.encoders = {col: {} for col in category_columns}
        self.next_code = {col: 0 for col in category_columns}
        
        print(f"Инициализирован OrdinalEncoder для столбцов: {category_columns}")
    
    def __call__(self, x: pd.Series) -> pd.Series:
        x_encoded = x.copy()
        
        for col in self.category_columns:
            if col in x_encoded.index:
                value = x_encoded[col]
                
                if value not in self.encoders[col]:
                    self.encoders[col][value] = self.next_code[col]
                    self.next_code[col] += 1
                
                x_encoded[col] = self.encoders[col][value]
        
        return x_encoded

categorical_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
ordinal_encoder = OrdinalEncoderTransform(categorical_columns)

bank_dataset_encoded = BankDataset(bank_df, transform=ordinal_encoder)

print(f"\nПример кодирования первого элемента:")
x_original, _ = bank_dataset[0]
x_encoded, _ = bank_dataset_encoded[0] 

print("\nИсходные категориальные признаки:")
for col in categorical_columns[:3]: 
    if col in x_original.index:
        print(f"{col}: {x_original[col]}")

print("\nЗакодированные признаки:")
for col in categorical_columns[:3]: 
    if col in x_encoded.index:
        print(f"{col}: {x_encoded[col]}")

print(f"\nСловари кодирования (пример для 'job'):")
print(ordinal_encoder.encoders['job'])

Инициализирован OrdinalEncoder для столбцов: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
Инициализирован BankDataset с 45211 образцами
- Используется transform: OrdinalEncoderTransform

Пример кодирования первого элемента:

Исходные категориальные признаки:
job: management
marital: married
education: tertiary

Закодированные признаки:
job: 0
marital: 0
education: 0

Словари кодирования (пример для 'job'):
{'management': 0}


<p class="task" id="4"></p>

4\. Опишите класс `LabelEncoderTransform`. Решение должно удовлетворять следующим критериям:

* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (строку) и возвращает целое число, соответствующее этой строке;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объекта в качестве аргумента `target_transform` объект класса `LabelEncoderTransform`.

- [ ] Проверено на семинаре

In [ ]:
class LabelEncoderTransform(Transform):
    """Кодировщик меток классов в числовые значения"""
    
    def __init__(self) -> None:
        self.label_to_code = {}
        self.next_code = 0
        
        print("Инициализирован LabelEncoderTransform")
    
    def __call__(self, x: str) -> int:
        if x not in self.label_to_code:
            self.label_to_code[x] = self.next_code
            self.next_code += 1
            print(f"Новая метка '{x}' получила код {self.label_to_code[x]}")
        
        return self.label_to_code[x]

label_encoder = LabelEncoderTransform()

bank_dataset_full_encoded = BankDataset(
    bank_df, 
    transform=OrdinalEncoderTransform(categorical_columns),
    target_transform=label_encoder
)

print(f"\nДемонстрация кодирования меток:")
for i in range(5):
    x, y = bank_dataset_full_encoded[i]
    original_y = bank_df['y'].iloc[i]
    print(f"Элемент {i}: исходная метка '{original_y}' -> код {y}")

print(f"\nСловарь кодирования меток: {label_encoder.label_to_code}")

Инициализирован LabelEncoderTransform
Инициализирован OrdinalEncoder для столбцов: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
Инициализирован BankDataset с 45211 образцами
- Используется transform: OrdinalEncoderTransform
- Используется target_transform: LabelEncoderTransform

Демонстрация кодирования меток:
Новая метка 'no' получила код 0
Элемент 0: исходная метка 'no' -> код 0
Элемент 1: исходная метка 'no' -> код 0
Элемент 2: исходная метка 'no' -> код 0
Элемент 3: исходная метка 'no' -> код 0
Элемент 4: исходная метка 'no' -> код 0

Словарь кодирования меток: {'no': 0}


<p class="task" id="5"></p>

5\. Опишите класс `ToTensor`.  Решение должно удовлетворять следующим критериям:
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает на вход серию или фрейм и возвращает тензор.

Опишите класс `Compose`.  Решение должно удовлетворять следующим критериям:
* при создании объекта в конструктор передается список объектов `transforms`, каждый из которых имеет метод `__call__(x, y)`;
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает имеет параметра (признаки и класс в числовом виде) и и возвращает кортеж, полученный путем последовательного вызова объектов из `transforms`.

Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании преобразования `Compose` список из объектов LabelEncoderTransform и ToTensor.

- [ ] Проверено на семинаре

In [7]:
class ToTensor(Transform):
    """Преобразование данных в тензоры PyTorch"""
    
    def __call__(self, X: pd.Series | int | float) -> torch.Tensor:
        if isinstance(X, pd.Series):
            try:
                return torch.FloatTensor(X.values.astype(float))
            except (ValueError, TypeError):
                numeric_values = []
                for val in X.values:
                    try:
                        numeric_values.append(float(val))
                    except (ValueError, TypeError):
                        numeric_values.append(0.0)
                return torch.FloatTensor(numeric_values)
        elif isinstance(X, (int, float)):
            return torch.tensor(X, dtype=torch.float32)
        else:
            return torch.tensor(float(X), dtype=torch.float32)

class FeatureCompose(Transform):
    """Композиция трансформаций для признаков (возвращает только признаки)"""
    
    def __init__(self, transforms: list[Transform]) -> None:
        self.transforms = transforms
        print(f"Инициализирована композиция из {len(transforms)} трансформаций")
        
    def __call__(self, X: Any) -> Any:
        result = X
        
        # Последовательно применяем каждую трансформацию к признакам
        for transform in self.transforms:
            result = transform(result)
        
        return result

to_tensor = ToTensor()

feature_transform = FeatureCompose([
    OrdinalEncoderTransform(categorical_columns),
    to_tensor
])

label_encoder_new = LabelEncoderTransform()

bank_dataset_tensor = BankDataset(
    bank_df,
    transform=feature_transform,
    target_transform=label_encoder_new
)

print(f"\nДемонстрация работы композиции трансформаций:")
x, y = bank_dataset_tensor[0]
print(f"Тип признаков: {type(x)}")
print(f"Форма тензора признаков: {x.shape}")
print(f"Первые 5 значений тензора: {x[:5]}")
print(f"Тип метки: {type(y)}")
print(f"Значение метки: {y}")

# Проверим несколько элементов
print(f"\nПроверка кодирования меток:")
for i in range(3):
    _, y_encoded = bank_dataset_tensor[i]
    y_original = bank_df['y'].iloc[i]
    print(f"'{y_original}' -> {y_encoded}")

Инициализирован OrdinalEncoder для столбцов: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
Инициализирована композиция из 2 трансформаций
Инициализирован LabelEncoderTransform
Инициализирован BankDataset с 45211 образцами
- Используется transform: FeatureCompose
- Используется target_transform: LabelEncoderTransform

Демонстрация работы композиции трансформаций:
Новая метка 'no' получила код 0
Тип признаков: <class 'torch.Tensor'>
Форма тензора признаков: torch.Size([16])
Первые 5 значений тензора: tensor([58.,  0.,  0.,  0.,  0.])
Тип метки: <class 'int'>
Значение метки: 0

Проверка кодирования меток:
'no' -> 0
'no' -> 0
'no' -> 0


<p class="task" id="6"></p>

6\. Разделите датасет из предыдущего задания на обучающую и тестовую выборку в соотношении 75% на 25%. Создайте объект `DataLoader` для получения пакетов размера 64, полученных из перемешанного обучающего датасета. Кастомизируйте `DataLoader` таким образом, чтобы пакет признаков был представлен в виде трехмерного тензора размера 64x2x8 (разделите 16 признаков на два тензора по 8). Получите один пакет и выведите на экран размерность тензоров пакета. 

- [ ] Проверено на семинаре

In [8]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

def custom_collate_fn(batch):


    features = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    

    features_tensor = torch.stack(features)  
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    

    batch_size, n_features = features_tensor.shape
    
    if n_features >= 16:
        features_reshaped = features_tensor[:, :16].view(batch_size, 2, 8)
    else:
        if n_features < 16:
            padding = torch.zeros(batch_size, 16 - n_features)
            features_padded = torch.cat([features_tensor, padding], dim=1)
            features_reshaped = features_padded.view(batch_size, 2, 8)
        else:
            features_reshaped = features_tensor[:, :16].view(batch_size, 2, 8)
    
    return features_reshaped, labels_tensor
train_indices, test_indices = train_test_split(
    range(len(bank_dataset_tensor)), 
    test_size=0.25, 
    random_state=42,
    stratify=[bank_df['y'].iloc[i] for i in range(len(bank_df))] 
)

print(f"Размер обучающей выборки: {len(train_indices)}")
print(f"Размер тестовой выборки: {len(test_indices)}")

train_dataset = Subset(bank_dataset_tensor, train_indices)
test_dataset = Subset(bank_dataset_tensor, test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,  
    collate_fn=custom_collate_fn,  
    drop_last=True 
)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,  
    collate_fn=custom_collate_fn,
    drop_last=False
)

print(f"\nСоздан DataLoader для обучения:")
print(f"- Размер батча: {train_loader.batch_size}")
print(f"- Перемешивание: True")
print(f"- Количество батчей: {len(train_loader)}")

print(f"\nПолучаем первый батч...")
batch_features, batch_labels = next(iter(train_loader))

print(f"\nРазмерности тензоров в батче:")
print(f"- Признаки: {batch_features.shape}  # (batch_size=64, groups=2, features_per_group=8)")
print(f"- Метки: {batch_labels.shape}      # (batch_size=64,)")

print(f"\nПример содержимого батча:")
print(f"- Первая группа признаков (8 признаков) для первого образца:")
print(f"  {batch_features[0, 0, :]}")
print(f"- Вторая группа признаков (8 признаков) для первого образца:")
print(f"  {batch_features[0, 1, :]}")
print(f"- Метки первых 10 образцов: {batch_labels[:10]}")

print(f"\nУникальные метки в батче: {torch.unique(batch_labels)}")
print(f"Количество образцов каждого класса в батче:")
for label in torch.unique(batch_labels):
    count = (batch_labels == label).sum().item()
    print(f"  Класс {label}: {count} образцов")

Размер обучающей выборки: 33908
Размер тестовой выборки: 11303

Создан DataLoader для обучения:
- Размер батча: 64
- Перемешивание: True
- Количество батчей: 529

Получаем первый батч...
Новая метка 'yes' получила код 1

Размерности тензоров в батче:
- Признаки: torch.Size([64, 2, 8])  # (batch_size=64, groups=2, features_per_group=8)
- Метки: torch.Size([64])      # (batch_size=64,)

Пример содержимого батча:
- Первая группа признаков (8 признаков) для первого образца:
  tensor([2.8000e+01, 3.0000e+00, 0.0000e+00, 2.0000e+00, 0.0000e+00, 3.2670e+03,
        0.0000e+00, 1.0000e+00])
- Вторая группа признаков (8 признаков) для первого образца:
  tensor([ 1., 18.,  0., 40.,  1., -1.,  0.,  0.])
- Метки первых 10 образцов: tensor([0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

Уникальные метки в батче: tensor([0, 1])
Количество образцов каждого класса в батче:
  Класс 0: 50 образцов
  Класс 1: 14 образцов
